# 🧠 Week 15 & 16: Deep Learning Fundamentals & Model Deployment

**SIT-N Hack-o-Week | 5th Semester (Third Year)**

---

### 📋 Syllabus Overview:
1. **Model Deployment Basics**: Saving and loading trained machine learning models using `pickle` and `joblib`.
2. **Perceptrons & Activation Functions**: Single-layer perceptrons, non-linear activation functions (Sigmoid, ReLU, Leaky ReLU, Tanh, Softmax), forward and backward propagation.
3. **Loss Functions & Optimizers**: Binary Cross-Entropy vs MSE, Stochastic Gradient Descent (SGD), Momentum, and Adam.
4. **Building a Neural Network from Scratch in NumPy**: Full matrix-based multi-layer perceptron implementation with forward propagation, analytical backpropagation, and parameter updates.

---

In [ ]:
# Setup & Core Imports
import os
import time
import pickle
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('Environment initialized successfully!')

## 1. 🗃️ Dataset Loading & Exploratory Preparation

We use the **Kaggle Student Performance & Academic Insights Dataset**. Our objective is to predict whether a student achieves a **Distinction** (`CGPA >= 8.5`).

In [ ]:
data_path = os.path.join('..', 'data', 'kaggle_student_performance.csv')
df = pd.read_csv(data_path)
df['Distinction'] = (df['CGPA'] >= 8.5).astype(int)

print(f"Dataset Shape: {df.shape}")
print(f"Distinction Class Distribution:\n{df['Distinction'].value_counts(normalize=True).round(3)}")
df.head()

## 2. 📦 Task 1: Intro to Model Deployment Basics (`pickle` vs `joblib`)

In machine learning engineering, training a model is only the first step. To deploy a model to production (e.g. inside a Flask REST API, microservice, or serverless function), we must serialize both the **learned weights** and the **preprocessing transformations** into a persistent binary file.

### Comparing Serialization Technologies:
| Feature | Python `pickle` | Scikit-Learn `joblib` |
|---|---|---|
| **Origin** | Python Standard Library | Scikit-Learn / SciPy Ecosystem |
| **Large NumPy Arrays** | Slower; copies data in memory | Highly optimized, memory-mapped I/O |
| **Compression** | Manual stream wrapping needed | Built-in transparent compression (zlib, gzip) |
| **Dependencies** | None (Built-in) | `pip install joblib` |
| **Security** | Arbitrary bytecode execution risk | Same risk as pickle; verify model origin |

In [ ]:
# Define Feature Columns
NUMERIC_FEATURES = ['Age', 'StudyHoursPerWeek', 'AttendanceRate', 'MathScore', 'ReadingScore', 'WritingScore']
CATEGORICAL_FEATURES = ['Gender', 'Department']

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df['Distinction'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Build Complete End-to-End Pipeline
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), CATEGORICAL_FEATURES)
])

pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

pipeline.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, pipeline.predict(X_test))
print(f"Pipeline fitted successfully! Test Accuracy: {baseline_acc * 100:.2f}%")

In [ ]:
# Benchmarking Serialization: Pickle vs Joblib
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)
pickle_file = os.path.join(models_dir, 'pipeline_pickle.pkl')
joblib_file = os.path.join(models_dir, 'pipeline_joblib.joblib')

# 1. Pickle Save
t0 = time.perf_counter()
with open(pickle_file, 'wb') as f:
    pickle.dump(pipeline, f, protocol=pickle.HIGHEST_PROTOCOL)
pickle_save_ms = (time.perf_counter() - t0) * 1000

# 2. Joblib Save (with compression=3)
t0 = time.perf_counter()
joblib.dump(pipeline, joblib_file, compress=3)
joblib_save_ms = (time.perf_counter() - t0) * 1000

# 3. Pickle Load
t0 = time.perf_counter()
with open(pickle_file, 'rb') as f:
    loaded_pickle = pickle.load(f)
pickle_load_ms = (time.perf_counter() - t0) * 1000

# 4. Joblib Load
t0 = time.perf_counter()
loaded_joblib = joblib.load(joblib_file)
joblib_load_ms = (time.perf_counter() - t0) * 1000

pickle_kb = os.path.getsize(pickle_file) / 1024
joblib_kb = os.path.getsize(joblib_file) / 1024

# Verify predictions match identically
assert np.array_equal(pipeline.predict(X_test), loaded_pickle.predict(X_test))
assert np.array_equal(pipeline.predict(X_test), loaded_joblib.predict(X_test))

benchmark_df = pd.DataFrame({
    'Metric': ['File Size (KB)', 'Save Latency (ms)', 'Load Latency (ms)', 'Integrity Verified'],
    'Pickle (.pkl)': [f"{pickle_kb:.2f} KB", f"{pickle_save_ms:.2f} ms", f"{pickle_load_ms:.2f} ms", 'True'],
    'Joblib (.joblib)': [f"{joblib_kb:.2f} KB", f"{joblib_save_ms:.2f} ms", f"{joblib_load_ms:.2f} ms", 'True']
})
benchmark_df

In [ ]:
# Live Deployment Inference Test
def predict_student_production(student_dict, backend='joblib'):
    """Production inference serving function loaded on-demand."""
    model_path = os.path.join(models_dir, f'pipeline_{backend}.{"joblib" if backend == "joblib" else "pkl"}')
    t0 = time.perf_counter()
    if backend == 'joblib':
        model = joblib.load(model_path)
    else:
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
    load_ms = (time.perf_counter() - t0) * 1000
    
    input_df = pd.DataFrame([student_dict])
    t_inf = time.perf_counter()
    pred = int(model.predict(input_df)[0])
    prob = model.predict_proba(input_df)[0]
    inf_ms = (time.perf_counter() - t_inf) * 1000
    
    return {
        'prediction': 'Distinction (CGPA >= 8.5)' if pred == 1 else 'Standard (CGPA < 8.5)',
        'probability_distinction': f"{prob[1]*100:.2f}%\开拓",
        'load_latency_ms': f"{load_ms:.2f} ms",
        'inference_latency_ms': f"{inf_ms:.2f} ms"
    }

sample_student = {
    'Age': 21,
    'Gender': 'Female',
    'Department': 'Computer Science',
    'StudyHoursPerWeek': 22.0,
    'AttendanceRate': 92.0,
    'MathScore': 88.0,
    'ReadingScore': 85.0,
    'WritingScore': 90.0
}
print("Live Production Serving Result:")
predict_student_production(sample_student, backend='joblib')

## 3. 🧠 Task 2: Perceptrons, Activation Functions & Propagation

### 3.1 The Rosenblatt Perceptron
The Perceptron (Frank Rosenblatt, 1957) is the atomic unit of artificial neural networks:
$$z = \sum_{i=1}^n w_i x_i + b = \mathbf{w}^T \mathbf{x} + b$$
$$\hat{y} = \begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{if } z < 0 \end{cases}$$

The perceptron updates weights only when an error occurs:
$$w_i \leftarrow w_i + \eta (y - \hat{y}) x_i$$
$$b \leftarrow b + \eta (y - \hat{y})$$

In [ ]:
class SinglePerceptron:
    def __init__(self, lr=0.1, epochs=20):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0
        self.errors_ = []

    def fit(self, X, y):
        self.weights = np.zeros(X.shape[1])
        self.bias = 0.0
        self.errors_ = []
        for _ in range(self.epochs):
            errors = 0
            for xi, target in zip(X, y):
                update = self.lr * (target - self.predict(xi))
                self.weights += update * xi
                self.bias += update
                errors += int(update != 0.0)
            self.errors_.append(errors)
        return self

    def net_input(self, X):
        return np.dot(X, self.weights) + self.bias

    def predict(self, X):
        return np.where(self.net_input(X) >= 0.0, 1, 0)

# Testing on Logical AND Gate
X_and = np.array([[0,0], [0,1], [1,0], [1,1]])
y_and = np.array([0, 0, 0, 1])
p_and = SinglePerceptron(lr=0.1, epochs=10).fit(X_and, y_and)
print(f"AND Gate Predictions: {p_and.predict(X_and)} (Target: {y_and})")

### 3.2 Activation Functions and Their Analytical Derivatives
Without non-linear activation functions, any multi-layer neural network collapses into a single linear transformation (since compositions of linear functions are linear).

1. **Sigmoid**: $\sigma(z) = \frac{1}{1 + e^{-z}}$, $\sigma'(z) = \sigma(z)(1 - \sigma(z))$
2. **ReLU**: $f(z) = \max(0, z)$, $f'(z) = 1$ if $z > 0$ else $0$
3. **Leaky ReLU**: $f(z) = \max(\alpha z, z)$, $f'(z) = 1$ if $z > 0$ else $\alpha$
4. **Tanh**: $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$, $\tanh'(z) = 1 - \tanh^2(z)$

In [ ]:
z = np.linspace(-5, 5, 200)

# Functions
sigmoid = 1.0 / (1.0 + np.exp(-z))
sigmoid_grad = sigmoid * (1.0 - sigmoid)

relu = np.maximum(0.0, z)
relu_grad = np.where(z > 0, 1.0, 0.0)

leaky = np.where(z > 0, z, 0.1 * z)
leaky_grad = np.where(z > 0, 1.0, 0.1)

tanh = np.tanh(z)
tanh_grad = 1.0 - tanh**2

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(z, sigmoid, label='Sigmoid $\\sigma(z)$', color='#8b5cf6', lw=2)
axes[0, 0].plot(z, sigmoid_grad, label="Derivative $\\sigma'(z)$", color='#06b6d4', linestyle='--', lw=2)
axes[0, 0].set_title('Sigmoid (Max grad = 0.25 -> Vanishing Gradient)')
axes[0, 0].legend()

axes[0, 1].plot(z, relu, label='ReLU $\\max(0, z)$', color='#3b82f6', lw=2)
axes[0, 1].plot(z, relu_grad, label="Derivative", color='#10b981', linestyle='--', lw=2)
axes[0, 1].set_title('ReLU (No vanishing grad for z > 0, zero for z < 0)')
axes[0, 1].legend()

axes[1, 0].plot(z, leaky, label='Leaky ReLU ($\\alpha=0.1$)', color='#f59e0b', lw=2)
axes[1, 0].plot(z, leaky_grad, label="Derivative", color='#ef4444', linestyle='--', lw=2)
axes[1, 0].set_title('Leaky ReLU (Prevents Dying Neurons)')
axes[1, 0].legend()

axes[1, 1].plot(z, tanh, label='Tanh $\\tanh(z)$', color='#ec4899', lw=2)
axes[1, 1].plot(z, tanh_grad, label="Derivative", color='#8b5cf6', linestyle='--', lw=2)
axes[1, 1].set_title('Tanh (Zero-centered between -1 and 1)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. 🎯 Task 3: Loss Functions & Optimizers (SGD vs Momentum vs Adam)

### Loss Functions:
1. **Binary Cross-Entropy (BCE) Loss**:
$$L(y, \hat{y}) = -\frac{1}{m} \sum_{i=1}^m \left[ y^{(i)} \log \hat{y}^{(i)} + (1 - y^{(i)}) \log (1 - \hat{y}^{(i)}) \right]$$
When coupled with a Sigmoid output $\hat{y} = \sigma(z)$, the derivative simplifies elegantly:
$$\frac{\partial L}{\partial z} = \hat{y} - y$$

### Optimizers:
1. **SGD**: $\theta \leftarrow \theta - \alpha \nabla_\theta L$
2. **Momentum**: $v \leftarrow \beta v + (1-\beta)\nabla_\theta L, \quad \theta \leftarrow \theta - \alpha v$
3. **Adam**: Exponential moving averages of first ($m_t$) and second ($v_t$) moments with bias correction:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

In [ ]:
# Simulating Optimization on an ill-conditioned ravine loss surface: f(x, y) = 0.5 * (x^2 + 10 * y^2)
def ravine_loss(x, y):
    return 0.5 * (x**2 + 10.0 * (y**2))

def ravine_grad(x, y):
    return np.array([x, 10.0 * y])

start = np.array([-4.0, 3.0])
steps = 35

# 1. SGD
curr_sgd = start.copy()
path_sgd = [curr_sgd.copy()]
for _ in range(steps):
    curr_sgd -= 0.15 * ravine_grad(curr_sgd[0], curr_sgd[1])
    path_sgd.append(curr_sgd.copy())
path_sgd = np.array(path_sgd)

# 2. Momentum
curr_mom = start.copy()
v_mom = np.zeros(2)
path_mom = [curr_mom.copy()]
for _ in range(steps):
    g = ravine_grad(curr_mom[0], curr_mom[1])
    v_mom = 0.9 * v_mom + 0.1 * g
    curr_mom -= 0.15 * v_mom
    path_mom.append(curr_mom.copy())
path_mom = np.array(path_mom)

# 3. Adam
curr_adam = start.copy()
m_adam = np.zeros(2)
v_adam = np.zeros(2)
path_adam = [curr_adam.copy()]
for t in range(1, steps + 1):
    g = ravine_grad(curr_adam[0], curr_adam[1])
    m_adam = 0.9 * m_adam + 0.1 * g
    v_adam = 0.999 * v_adam + 0.001 * (g**2)
    m_hat = m_adam / (1.0 - 0.9**t)
    v_hat = v_adam / (1.0 - 0.999**t)
    curr_adam -= (0.35 / (np.sqrt(v_hat) + 1e-8)) * m_hat
    path_adam.append(curr_adam.copy())
path_adam = np.array(path_adam)

# Plot Trajectories
plt.figure(figsize=(10, 6))
x_vals = np.linspace(-5, 5, 100)
y_vals = np.linspace(-4, 4, 100)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
Z_grid = ravine_loss(X_grid, Y_grid)

plt.contour(X_grid, Y_grid, Z_grid, levels=25, cmap='viridis', alpha=0.5)
plt.plot(path_sgd[:, 0], path_sgd[:, 1], 'o-', label='Vanilla SGD', color='#3b82f6', lw=2)
plt.plot(path_mom[:, 0], path_mom[:, 1], 's-', label='SGD + Momentum (\\beta=0.9)', color='#10b981', lw=2)
plt.plot(path_adam[:, 0], path_adam[:, 1], '^-', label='Adam Optimizer', color='#8b5cf6', lw=2.5)
plt.plot(0, 0, 'r*', markersize=16, label='Global Minimum (0,0)')
plt.title('Optimization Paths on Anisotropic Loss Ravine')
plt.xlabel('Parameter X')
plt.ylabel('Parameter Y')
plt.legend()
plt.show()

## 5. ⚡ Task 4: Building a Neural Network from Scratch in Pure NumPy

We now engineer a modular deep neural network architecture from scratch using pure NumPy:
- **Weight Initialization**: He Normal initialization for ReLU layers ($W \sim \mathcal{N}(0, \sqrt{2 / n_{in}})$) to prevent exploding/vanishing activations.
- **Forward Propagation**: Iterative matrix multiplications $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$ and activation caching.
- **Backpropagation**: Analytical application of chain rule to calculate $\frac{\partial L}{\partial W^{[l]}}$ and $\frac{\partial L}{\partial b^{[l]}}$.
- **Optimizers**: Integrated SGD and Adam parameter updates.

In [ ]:
class NumPyMLP:
    def __init__(self, layer_dims, activation='relu', optimizer='adam', lr=0.03):
        self.layer_dims = layer_dims
        self.activation = activation.lower()
        self.optimizer = optimizer.lower()
        self.lr = lr
        self.params = {}
        self.m = {}
        self.v = {}
        self.t = 0
        self._init_params()

    def _init_params(self):
        np.random.seed(42)
        for l in range(1, len(self.layer_dims)):
            n_in = self.layer_dims[l - 1]
            n_out = self.layer_dims[l]
            scale = np.sqrt(2.0 / n_in) if self.activation == 'relu' else np.sqrt(1.0 / n_in)
            self.params[f'W{l}'] = np.random.randn(n_out, n_in) * scale
            self.params[f'b{l}'] = np.zeros((n_out, 1))
            self.m[f'dW{l}'] = np.zeros_like(self.params[f'W{l}'])
            self.m[f'db{l}'] = np.zeros_like(self.params[f'b{l}'])
            self.v[f'dW{l}'] = np.zeros_like(self.params[f'W{l}'])
            self.v[f'db{l}'] = np.zeros_like(self.params[f'b{l}'])

    def _act(self, Z, name):
        if name == 'relu':
            return np.maximum(0.0, Z)
        elif name == 'sigmoid':
            return 1.0 / (1.0 + np.exp(-np.clip(Z, -25, 25)))
        elif name == 'tanh':
            return np.tanh(Z)
        return Z

    def _act_deriv(self, Z, name):
        if name == 'relu':
            return np.where(Z > 0, 1.0, 0.0)
        elif name == 'sigmoid':
            s = self._act(Z, 'sigmoid')
            return s * (1.0 - s)
        elif name == 'tanh':
            return 1.0 - np.tanh(Z)**2
        return np.ones_like(Z)

    def forward(self, X):
        caches = []
        A = X
        L = len(self.layer_dims) - 1
        for l in range(1, L):
            A_prev = A
            Z = np.dot(self.params[f'W{l}'], A_prev) + self.params[f'b{l}']
            A = self._act(Z, self.activation)
            caches.append((A_prev, self.params[f'W{l}'], self.params[f'b{l}'], Z, self.activation))
        # Output layer (Sigmoid)
        A_prev = A
        Z = np.dot(self.params[f'W{L}'], A_prev) + self.params[f'b{L}']
        A = self._act(Z, 'sigmoid')
        caches.append((A_prev, self.params[f'W{L}'], self.params[f'b{L}'], Z, 'sigmoid'))
        return A, caches

    def backward(self, AL, Y, caches):
        grads = {}
        m = Y.shape[1]
        L = len(caches)
        dZ = AL - Y
        A_prev, W, b, Z, act = caches[L - 1]
        grads[f'dW{L}'] = (1.0 / m) * np.dot(dZ, A_prev.T)
        grads[f'db{L}'] = (1.0 / m) * np.sum(dZ, axis=1, keepdims=True)
        dA_prev = np.dot(W.T, dZ)

        for l in reversed(range(1, L)):
            A_prev, W, b, Z, act = caches[l - 1]
            dZ = dA_prev * self._act_deriv(Z, act)
            grads[f'dW{l}'] = (1.0 / m) * np.dot(dZ, A_prev.T)
            grads[f'db{l}'] = (1.0 / m) * np.sum(dZ, axis=1, keepdims=True)
            if l > 1:
                dA_prev = np.dot(W.T, dZ)
        return grads

    def update_params(self, grads):
        L = len(self.layer_dims) - 1
        self.t += 1
        if self.optimizer == 'adam':
            beta1, beta2, eps = 0.9, 0.999, 1e-8
            for l in range(1, L + 1):
                self.m[f'dW{l}'] = beta1 * self.m[f'dW{l}'] + (1 - beta1) * grads[f'dW{l}']
                self.m[f'db{l}'] = beta1 * self.m[f'db{l}'] + (1 - beta1) * grads[f'db{l}']
                self.v[f'dW{l}'] = beta2 * self.v[f'dW{l}'] + (1 - beta2) * (grads[f'dW{l}']**2)
                self.v[f'db{l}'] = beta2 * self.v[f'db{l}'] + (1 - beta2) * (grads[f'db{l}']**2)
                m_corr_w = self.m[f'dW{l}'] / (1 - beta1**self.t)
                m_corr_b = self.m[f'db{l}'] / (1 - beta1**self.t)
                v_corr_w = self.v[f'dW{l}'] / (1 - beta2**self.t)
                v_corr_b = self.v[f'db{l}'] / (1 - beta2**self.t)
                self.params[f'W{l}'] -= (self.lr / (np.sqrt(v_corr_w) + eps)) * m_corr_w
                self.params[f'b{l}'] -= (self.lr / (np.sqrt(v_corr_b) + eps)) * m_corr_b
        else:
            for l in range(1, L + 1):
                self.params[f'W{l}'] -= self.lr * grads[f'dW{l}']
                self.params[f'b{l}'] -= self.lr * grads[f'db{l}']

    def fit(self, X, y, epochs=150, X_val=None, y_val=None):
        history = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': []}
        for epoch in range(1, epochs + 1):
            AL, caches = self.forward(X)
            cost = -np.mean(y * np.log(np.clip(AL, 1e-12, 1-1e-12)) + (1-y) * np.log(np.clip(1-AL, 1e-12, 1-1e-12)))
            grads = self.backward(AL, y, caches)
            self.update_params(grads)
            
            acc = np.mean((AL >= 0.5) == y) * 100
            history['loss'].append(cost)
            history['acc'].append(acc)
            
            if X_val is not None:
                AL_val, _ = self.forward(X_val)
                val_cost = -np.mean(y_val * np.log(np.clip(AL_val, 1e-12, 1-1e-12)) + (1-y_val) * np.log(np.clip(1-AL_val, 1e-12, 1-1e-12)))
                val_acc = np.mean((AL_val >= 0.5) == y_val) * 100
                history['val_loss'].append(val_cost)
                history['val_acc'].append(val_acc)
        return history

    def predict(self, X):
        AL, _ = self.forward(X)
        return (AL >= 0.5).astype(int)

In [ ]:
# Train NumPy MLP on Kaggle Student Performance Dataset
from sklearn.preprocessing import StandardScaler
X_proc = preprocessor.fit_transform(X)
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_proc, y, test_size=0.25, random_state=42, stratify=y)

# Transpose into (n_features, n_samples) matrix form for our NumPy Engine
X_train_T = X_train_p.T
y_train_T = y_train_p.reshape(1, -1)
X_test_T = X_test_p.T
y_test_T = y_test_p.reshape(1, -1)

mlp = NumPyMLP(layer_dims=[X_train_T.shape[0], 16, 8, 1], activation='relu', optimizer='adam', lr=0.03)
history = mlp.fit(X_train_T, y_train_T, epochs=150, X_val=X_test_T, y_val=y_test_T)

preds = mlp.predict(X_test_T)[0]
print(f"Scratch NumPy MLP Test Accuracy: {accuracy_score(y_test_p, preds)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test_p, preds):.4f}")
print(classification_report(y_test_p, preds, target_names=['Standard', 'Distinction']))

In [ ]:
# Plot Training & Validation Loss/Accuracy Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['loss'], label='Train Loss (BCE)', color='#8b5cf6', lw=2)
ax1.plot(history['val_loss'], label='Val Loss (BCE)', color='#f43f5e', linestyle='--', lw=2)
ax1.set_title('NumPy MLP: Binary Cross-Entropy Loss Progression')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history['acc'], label='Train Accuracy (%)', color='#06b6d4', lw=2)
ax2.plot(history['val_acc'], label='Val Accuracy (%)', color='#10b981', linestyle='--', lw=2)
ax2.set_title('NumPy MLP: Accuracy Progression')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. 🏁 Summary of Key Takeaways

1. **Model Deployment**:
   - `joblib` is the gold standard serialization framework for Scikit-Learn / NumPy models, delivering up to 90% disk space reduction and faster cold start times compared to raw `pickle`.
   - Always persist the full end-to-end `Pipeline` to prevent preprocessor drift in production.
2. **Activation Functions & Gradient Dynamics**:
   - Sigmoid and Tanh squish activations, but their derivative tops out at $\le 0.25$, creating vanishing gradients across deep architectures.
   - ReLU ensures clean gradient flow ($f'=1$) for positive values, while Leaky ReLU eliminates the "dying neuron" pathology.
3. **Optimization Efficiency**:
   - Vanilla SGD oscillates along high-curvature ravines.
   - Momentum dampens orthogonal noise, and **Adam** adaptively scales per-parameter step sizes, achieving the fastest convergence.
4. **NumPy Implementation**:
   - Implementing neural networks directly in matrix form deepens intuition for tensor contractions, matrix calculus, and automated learning systems.